# DATA209 — Advanced Exploratory Data Analysis
# Practical P25-26 · Feature engineering and selection

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 13 · Module 4 · CO4

---

**Objective.** Construct features that encode domain knowledge, then compare filter, wrapper and embedded selection methods on the result.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 to P23-24 — cleaned data with the skewed columns log-transformed.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# --- from P15-16
df_clean = df.drop_duplicates().reset_index(drop=True)
for c in ["VisitorType", "Month"]:
    df_clean[c] = df_clean[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

model_cols = ["Administrative", "Administrative_Duration", "Informational",
              "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
              "BounceRates", "ExitRates", "PageValues"]

# --- from P23-24: log1p applied to the strongly skewed columns
skew_before = df_clean[model_cols].skew().sort_values(ascending=False)
needs = skew_before[skew_before.abs() > 1].index.tolist()
df_t = df_clean.copy()
for c in needs:
    df_t[c] = np.log1p(df_t[c].clip(lower=0))

print('log-transformed:', needs)

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P25-26 — Feature engineering and selection

### Feature engineering exercises

A feature is a question you have decided to ask of every row. Each constructed feature below
carries a one-line rationale — build nothing you cannot justify.

In [ ]:
# ---- Construct features -------------------------------------------------
fe = df_t.copy()

# 1 — total session depth and duration: overall engagement
fe["total_pages"]    = fe["Administrative"] + fe["Informational"] + fe["ProductRelated"]
fe["total_duration"] = (fe["Administrative_Duration"] + fe["Informational_Duration"]
                        + fe["ProductRelated_Duration"])

# 2 — average time per page: intent quality rather than raw volume
fe["avg_time_per_page"] = fe["total_duration"] / fe["total_pages"].replace(0, np.nan)
fe["avg_time_per_page"] = fe["avg_time_per_page"].fillna(0)

# 3 — product focus: what share of the session was spent on product pages
fe["product_focus"] = fe["ProductRelated"] / fe["total_pages"].replace(0, np.nan)
fe["product_focus"] = fe["product_focus"].fillna(0)

# 4 — engagement gap: exit rate relative to bounce rate
fe["exit_bounce_gap"] = fe["ExitRates"] - fe["BounceRates"]

# 5 — indicators: cheap, interpretable, often strong
fe["has_page_value"]   = (df_clean["PageValues"] > 0).astype(int)
fe["is_returning"]     = (fe["VisitorType"] == "Returning_Visitor").astype(int)
fe["is_holiday_month"] = fe["Month"].isin(["Nov", "Dec"]).astype(int)

# 6 — interaction: value only realised when the visitor engages
fe["value_x_depth"] = df_clean["PageValues"] * fe["ProductRelated"]

new_features = ["total_pages", "total_duration", "avg_time_per_page", "product_focus",
                "exit_bounce_gap", "has_page_value", "is_returning",
                "is_holiday_month", "value_x_depth"]

rationale = {
    "total_pages"      : "Overall session depth across all page types",
    "total_duration"   : "Overall time invested in the session",
    "avg_time_per_page": "Quality of attention rather than raw volume",
    "product_focus"    : "Share of the session spent on product pages",
    "exit_bounce_gap"  : "Sessions that continued after landing",
    "has_page_value"   : "Did the session reach a page on the checkout path",
    "is_returning"     : "Prior familiarity with the retailer",
    "is_holiday_month" : "November-December seasonal peak",
    "value_x_depth"    : "Page value only converts when the visitor browses",
}
print(pd.DataFrame({"feature": new_features,
                    "rationale": [rationale[f] for f in new_features]}).to_string(index=False))

print("\nAssociation of each new feature with the target:")
corrs = fe[new_features].corrwith(fe[TARGET].astype(int)).sort_values(key=abs, ascending=False)
print(corrs.round(3).to_string())

In [ ]:
# ---- Assemble the modelling matrix --------------------------------------
feature_cols = model_cols + new_features
Xf = fe[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
yf = fe[TARGET].astype(int)

print("Feature matrix:", Xf.shape)
print("Target balance :", f"{yf.mean()*100:.2f}% positive")

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(
    Xf, yf, test_size=0.2, stratify=yf, random_state=RANDOM_STATE)
print(f"Train {X_tr.shape}  Test {X_te.shape}")
print("Selection is fitted on the TRAINING data only — see P29-30 on leakage.")

### Feature selection comparison

Three families, three different questions. **Agreement between them is the signal worth trusting.**

In [ ]:
# ---- Filter, wrapper, embedded ------------------------------------------
from sklearn.feature_selection import (SelectKBest, f_classif,
                                       mutual_info_classif, RFE)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

K = 8
scaler_sel = StandardScaler().fit(X_tr)
X_tr_s = scaler_sel.transform(X_tr)

# filter 1 — ANOVA F
anova = SelectKBest(f_classif, k=K).fit(X_tr_s, y_tr)
# filter 2 — mutual information (catches non-linear dependence)
mi = SelectKBest(mutual_info_classif, k=K).fit(X_tr_s, y_tr)
# wrapper — recursive feature elimination
rfe = RFE(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
          n_features_to_select=K).fit(X_tr_s, y_tr)
# embedded 1 — random forest importance
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE,
                            n_jobs=-1).fit(X_tr, y_tr)
# embedded 2 — L1 logistic regression
lasso = LogisticRegression(penalty="l1", solver="liblinear", C=0.1,
                           max_iter=2000, random_state=RANDOM_STATE).fit(X_tr_s, y_tr)

sel = pd.DataFrame({
    "anova_F"   : anova.scores_,
    "mutual_info": mi.scores_,
    "rf_importance": rf.feature_importances_,
    "lasso_coef": np.abs(lasso.coef_.ravel()),
    "picked_anova": anova.get_support(),
    "picked_mi"   : mi.get_support(),
    "picked_rfe"  : rfe.support_,
}, index=feature_cols)

sel["picked_rf"]    = sel["rf_importance"].rank(ascending=False) <= K
sel["picked_lasso"] = sel["lasso_coef"] > 0
sel["votes"] = sel[["picked_anova", "picked_mi", "picked_rfe",
                    "picked_rf", "picked_lasso"]].sum(axis=1)

print(sel.sort_values("votes", ascending=False).round(4).to_string())

In [ ]:
# ---- Where do the methods agree? ---------------------------------------
consensus = sel[sel["votes"] >= 4].index.tolist()
contested = sel[(sel["votes"] > 0) & (sel["votes"] < 3)].index.tolist()

print(f"Chosen by 4 or 5 of 5 methods ({len(consensus)}):")
for f in consensus: print("   -", f)
print(f"\nChosen by only one or two methods ({len(contested)}) — treat with suspicion:")
for f in contested: print("   -", f)

plt.figure(figsize=(9, 4.4))
sns.barplot(x=sel["votes"].sort_values(ascending=False),
            y=sel["votes"].sort_values(ascending=False).index, color="#6B4C7A")
plt.xlabel("number of selection methods that chose the feature"); plt.ylabel("")
plt.title("Agreement across filter, wrapper and embedded selection")
plt.tight_layout(); plt.show()

engineered_kept = [f for f in consensus if f in new_features]
print(f"\n{len(engineered_kept)} of {len(new_features)} engineered features survived selection:")
print("  ", engineered_kept)
print("\nThat is the test of feature engineering — did the new features earn their place?")

In [ ]:
# ---- Does selection actually help? --------------------------------------
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def score_subset(cols, label):
    pipe = Pipeline([("scale", StandardScaler()),
                     ("model", LogisticRegression(max_iter=2000,
                                                  random_state=RANDOM_STATE))])
    s = cross_val_score(pipe, X_tr[cols], y_tr, cv=cv, scoring="roc_auc", n_jobs=-1)
    return {"feature set": label, "n_features": len(cols),
            "roc_auc_mean": s.mean(), "roc_auc_std": s.std()}

results = pd.DataFrame([
    score_subset(feature_cols, "all features"),
    score_subset(model_cols, "original only"),
    score_subset(consensus, "consensus selection"),
    score_subset(sel["rf_importance"].nlargest(5).index.tolist(), "top 5 by RF importance"),
]).set_index("feature set")
print(results.round(4).to_string())

print("\nInterpretation")
print("- A much smaller feature set that performs comparably is the better model:")
print("  it is cheaper, more stable and easier to explain.")
print("- Report the spread as well as the mean. A high mean with a wide spread is unstable.")

### Deliverable — P25-26

A notebook with the **engineered-feature table and rationales**, the five-method comparison, the
agreement chart, and a shortlist justified by method agreement rather than by a single score.